# Site CDR

Reads the `met_forcing_rxn` MIN3P runs off S3 and writes
`site_cdr.zarr`, dims `(site 8, case 4, plane 4, time 3651)`, to
`s3://carbonplan-carbon-removal/ew-workflows-data/min3p/simulations/postprocessed/met_forcing_rxn/`.

| variable | dims | what it is |
|---|---|---|
| `cdr` | `(site, case, plane, time)` | cumulative CDR, t CO2/ha, **carbonate-corrected** |
| `cdr_alk` | `(site, case, plane, time)` | the same at a flat 1 mol CO2/eq, uncorrected |
| `carbonate_c` | `(site, case, plane, time)` | the correction, `cdr = cdr_alk - carbonate_c` |
| `alk_export_ctrl`, `alk_export_erw` | `(site, case, plane, time)` | the gross alkalinity terms `cdr_alk` differences |
| `drainage_ctrl`, `drainage_erw` | `(site, case, plane, time)` | cumulative water past the plane, m |
| `forsterite_mass` | `(site, case, time)` | feedstock forsterite left in the column, t/ha |
| `cdr_potential` | `(site,)` | CDR at complete conversion of the feedstock, t CO2/ha |

CDR is the ERW minus CTRL cumulative alkalinity export across a control plane, at 44.01 g/mol,
less the carbon that came out of the soil's own carbonate. 

## 0. Setup

In [1]:
import io
import os
import re
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import s3fs
import xarray as xr

from byte_util.util import all_sites, calc_specific_discharge, states_per_site

# ----------------------------------------------------------------- knobs
WRITE = True          # False builds everything in memory but writes nothing to S3
OVERWRITE = True      # mode="w" on the zarr store; False refuses to clobber
WORKERS = 8           # parallel S3 downloads
PLANE = 100           # the plane summarised in the printouts; `plane` is a dimension
# ------------------------------------------------------------------------

SITES = list(all_sites)
CASES = ["hourly", "daily", "monthly", "longterm"]   # ordered coarse-ward
TREATMENTS = ["ctrl", "erw"]                         # `cdr` is erw - ctrl
PLANES = [50, 100, 200, 300]                         # every plane with a monitored cell pair
RUN_DAYS = 3650
SURFACE_CV = 401          # control volume at the ground surface, on a 401-cell 1 cm grid
DOMAIN_TOP_M = 4.00       # elevation of that cell

# Mg2SiO4 + 4 CO2 + 4 H2O -> 2 Mg+2 + 4 HCO3- + H4SiO4, from `simulations/databases/mineral.dbs`.
MINERAL = "forst-ph"
EQ_PER_MOL_FORST = 4.0
MOLAR_MASS = 140.6931                     # g/mol, forsterite
MW_CO2 = 44.01                            # g/mol
# conversions: eq/m2 -> t CO2/ha at 1 mol CO2 per eq, and mol CO2/m2 -> t CO2/ha.
EQ_TO_T_CO2_HA = MW_CO2 * 1e-6 * 1e4
MOL_TO_T_HA = MOLAR_MASS * 1e-6 * 1e4     # mol/m2 -> t/ha

# CaCO3 + CO2 + H2O -> Ca+2 + 2 HCO3-: 2 eq of alkalinity but only 1 mol of CO2 consumed.
CALCITE = "calcite-ph"
V_CALCITE = 100.0894 / 2.710 * 1e-6       # m3/mol, gfw / density from `mineral.dbs`
CARBON_PER_MOL_CALCITE = 1.0              # mol CO2 not removed, per mole of calcite dissolved

BUCKET_ROOT = ("carbonplan-carbon-removal/ew-workflows-data/min3p/simulations"
               "/met_forcing_rxn/min3p_runs")
OUT_ROOT = ("s3://carbonplan-carbon-removal/ew-workflows-data/min3p/simulations"
            "/postprocessed/met_forcing_rxn")

# Staged files are size-checked against S3 each pass, so a re-upload re-downloads.
CACHE = Path(os.environ.get("MIN3P_OUTPUT_CACHE", Path.home() / ".cache" / "min3p_outputs"))
STAGE = CACHE / "met_forcing_rxn_cdr"
STAGE.mkdir(parents=True, exist_ok=True)

DAYS = np.arange(0, RUN_DAYS + 1, dtype=float)
RUNS = [(s, c, t) for s in SITES for c in CASES for t in TREATMENTS]
fs = s3fs.S3FileSystem()

assert PLANE in PLANES, f"PLANE must be one of {PLANES}"
print(f"{len(RUNS)} runs: {len(SITES)} sites x {len(CASES)} cases x {len(TREATMENTS)} treatments")
print("staging into:", STAGE)

64 runs: 8 sites x 4 cases x 2 treatments
staging into: /Users/tylerkukla/.cache/min3p_outputs/met_forcing_rxn_cdr


## 1. Staging

Per run: `<case>.dat` (van Genuchten parameters and the monitor layout), the `.gbp` (heads) and
`.gbm` (alkalinity) files for the cells straddling each plane, and all 42 `.gsv` snapshots (mineral
volume fraction against depth) for the carbonate correction. About 200 MB of `.gsv` in total.

In [2]:
def run_key(site, case, treat):
    return f"{BUCKET_ROOT}/{site}/{case}_{treat}"


def run_dir(site, case, treat):
    return STAGE / site / f"{case}_{treat}"


def deck_monitor_cvs(text):
    """Control volumes of a deck's transient monitors, in file-index order (data block 8)."""
    lines = text.splitlines()
    i = next(k for k, ln in enumerate(lines) if "output of transient data" in ln.lower())

    def nums(line):
        return [int(x) for x in re.findall(r"\d+", line.split(";")[0])]   # `;` comments hold digits

    n = nums(lines[i + 1])[0]
    cvs, k = [], i + 3
    while len(cvs) < n:
        cvs += nums(lines[k])
        k += 1
    return cvs[:n]


WANTED_DEPTHS = sorted({d for cp in PLANES for d in (cp, cp + 1)})


def fetch_run(site, case, treat):
    """Stage one run; return (bytes fetched, monitor index -> depth in cm)."""
    local = run_dir(site, case, treat)
    local.mkdir(parents=True, exist_ok=True)
    infos = {i["name"].rsplit("/", 1)[-1]: i
             for i in fs.ls(run_key(site, case, treat), detail=True)}
    got = 0

    def pull(names):
        nonlocal got
        for name in names:
            info = infos.get(name)
            if info is None:
                raise FileNotFoundError(f"{run_key(site, case, treat)}/{name}")
            dest = local / name
            if dest.exists() and dest.stat().st_size == info["size"]:
                continue
            fs.get_file(info["name"], str(dest))
            got += info["size"]

    pull([f"{case}.dat"])
    deck = (local / f"{case}.dat").read_text(errors="replace")
    depths = {i + 1: SURFACE_CV - cv for i, cv in enumerate(deck_monitor_cvs(deck))}
    index = {d: i for i, d in depths.items()}
    missing = [d for d in WANTED_DEPTHS if d not in index]
    if missing:
        raise KeyError(f"{site}/{case}_{treat} monitors no cell at {missing} cm")

    pull([f"{case}_{index[d]}.{e}" for d in WANTED_DEPTHS for e in ("gbp", "gbm")])

    gsv = [n for n in infos if n.endswith(".gsv")]
    if not gsv:
        raise FileNotFoundError(f"{run_key(site, case, treat)} has no .gsv snapshots")
    pull(gsv)
    return got, depths


with ThreadPoolExecutor(WORKERS) as pool:
    staged = dict(zip(RUNS, pool.map(lambda r: fetch_run(*r), RUNS)))

layouts = {tuple(sorted(d.items())) for _, d in staged.values()}
assert len(layouts) == 1, f"runs disagree on their monitor layout ({len(layouts)} distinct)"
DEPTH_INDEX = {d: i for i, d in staged[RUNS[0]][1].items()}

print(f"{sum(g for g, _ in staged.values()) / 1e9:.2f} GB fetched this pass")
print("plane pairs -> file index:",
      {cp: (DEPTH_INDEX[cp], DEPTH_INDEX[cp + 1]) for cp in PLANES})

0.20 GB fetched this pass
plane pairs -> file index: {50: (8, 9), 100: (12, 17), 200: (18, 19), 300: (20, 21)}


In [3]:
def exit_status(site, case, treat):
    """The last banner MIN3P wrote, from a range read of the tail of `<case>_o.gen`."""
    text = fs.cat_file(f"{run_key(site, case, treat)}/{case}_o.gen", start=-2000).decode("latin1")
    if "normal exit" in text:
        return "normal exit"
    fail = re.search(r"\*+\s*(.*?exit.*?|failure.*?)\s*\*+", text, re.I)
    return fail.group(1).strip() if fail else "unknown"


with ThreadPoolExecutor(WORKERS) as pool:
    statuses = dict(zip(RUNS, pool.map(lambda r: exit_status(*r), RUNS)))
bad = {r: s for r, s in statuses.items() if s != "normal exit"}
assert not bad, f"{len(bad)} runs did not exit normally: {bad}"
print(f"all {len(RUNS)} runs exited normally")

all 64 runs exited normally


## 2. Readers

In [4]:
def parse_transient(text, label=""):
    """A MIN3P transient output file as a DataFrame, with `attrs['volume']` from the zone line.

    The `variables =` line wraps over several lines in `.gbm` and `_o.mms`.
    """
    lines = text.splitlines()
    zone = next(i for i, ln in enumerate(lines) if ln.lstrip().lower().startswith("zone"))
    v0 = next(i for i, ln in enumerate(lines) if ln.lstrip().lower().startswith("variables"))
    names = [n.strip() for n in re.findall(r'"([^"]*)"', "\n".join(lines[v0:zone]))]
    df = pd.read_csv(io.StringIO("\n".join(lines[zone + 1:])), sep=r"\s+", header=None)
    if df.shape[1] != len(names):
        raise ValueError(f"{label}: {len(names)} variable names, {df.shape[1]} columns")
    df = df.astype(float)             # raises rather than carrying strings through the arithmetic
    df.columns = names
    vol = re.search(r"volume\s*=\s*(\d+)", lines[zone])
    df.attrs["volume"] = int(vol.group(1)) if vol else None
    return df


def read_local(path):
    return parse_transient(Path(path).read_text(errors="replace"), str(path))


def dedupe(t, *arrays):
    """Drop repeated timestamps, keeping the last record written at each time."""
    t = np.asarray(t, dtype=float)
    keep = np.ones(t.size, dtype=bool)
    keep[:-1] = t[1:] != t[:-1]
    return (t[keep], *[np.asarray(a)[keep] for a in arrays])


def cumulative(t, f):
    """Trapezoidal running integral of an irregularly sampled rate."""
    out = np.zeros_like(f, dtype=float)
    out[1:] = np.cumsum(0.5 * (f[1:] + f[:-1]) * np.diff(t))
    return out

## 3. Alkalinity export across each control plane

Downward is positive. `.gbm` carries a *t* = 0 row that `.gbp` does not, so it is sliced off.

In [5]:
ALK_COL = "Alk [eq/L]"


def plane_export(site, case, treat):
    """Cumulative alkalinity export (eq/m2) and drainage (m) past each plane, on integer days."""
    folder = run_dir(site, case, treat)
    deck = folder / f"{case}.dat"
    gbp, gbm = {}, {}
    for d in WANTED_DEPTHS:
        i = DEPTH_INDEX[d]
        gbp[d] = read_local(folder / f"{case}_{i}.gbp")
        gbm[d] = read_local(folder / f"{case}_{i}.gbm")
        for fam, ext in ((gbp[d], "gbp"), (gbm[d], "gbm")):
            got = SURFACE_CV - fam.attrs["volume"]
            if got != d:
                raise ValueError(f"{folder}/{case}_{i}.{ext} is {got} cm, not {d} cm")

    pcols = list(gbp[WANTED_DEPTHS[0]].columns)
    t = gbp[PLANES[0]]["time"].to_numpy()
    alk_out, water_out = {}, {}
    for cp in PLANES:
        up, dn = cp, cp + 1
        stack = np.stack([gbp[up][pcols].to_numpy().T, gbp[dn][pcols].to_numpy().T])
        q = calc_specific_discharge(stack, input_file=deck, ci_a=0, ci_b=1,
                                    elev_a=DOMAIN_TOP_M - cp / 100,
                                    elev_b=DOMAIN_TOP_M - 0.01 - cp / 100,
                                    gbp_cols=pcols)
        upstream = gbp[up]["h_w"].to_numpy() > gbp[dn]["h_w"].to_numpy()
        alk = np.where(upstream,
                       gbm[up][ALK_COL].to_numpy()[1:],
                       gbm[dn][ALK_COL].to_numpy()[1:])
        rate = -q * alk * 1000.0                      # eq/L * m/d * 1000 L/m3 -> eq/m2/d
        td, r, w = dedupe(t, rate, -q)
        alk_out[cp] = np.interp(DAYS, td, cumulative(td, r))
        water_out[cp] = np.interp(DAYS, td, cumulative(td, w))
    return alk_out, water_out, len(t)


export, drained = {}, {}
for run in RUNS:
    export[run], drained[run], steps = plane_export(*run)
    print(f"  {run[0]:14s} {run[1]:9s} {run[2]:5s} {steps:7d} timesteps", flush=True)

  Cecil          hourly    ctrl    46397 timesteps
  Cecil          hourly    erw     46501 timesteps
  Cecil          daily     ctrl     3698 timesteps
  Cecil          daily     erw      3818 timesteps
  Cecil          monthly   ctrl     3673 timesteps
  Cecil          monthly   erw      3673 timesteps
  Cecil          longterm  ctrl     3682 timesteps
  Cecil          longterm  erw      3682 timesteps
  Flanagan       hourly    ctrl    44358 timesteps
  Flanagan       hourly    erw     44347 timesteps
  Flanagan       daily     ctrl     3121 timesteps
  Flanagan       daily     erw      3103 timesteps
  Flanagan       monthly   ctrl     3673 timesteps
  Flanagan       monthly   erw      3673 timesteps
  Flanagan       longterm  ctrl     3673 timesteps
  Flanagan       longterm  erw      3673 timesteps
  HoustonBlack   hourly    ctrl    45871 timesteps
  HoustonBlack   hourly    erw     46704 timesteps
  HoustonBlack   daily     ctrl     4361 timesteps
  HoustonBlack   daily     erw 

## 4. Calcite dissolved above each plane

Read from the `.gsv` snapshots: volume fraction x cell thickness above the plane, over calcite's
molar volume, differenced against snapshot 0. 

In [6]:
def calcite_dissolved(site, case, treat):
    """Calcite dissolved (+) or precipitated (-) above each plane, mol/m2, on integer days."""
    folder = run_dir(site, case, treat)
    files = sorted(folder.glob(f"{case}_*.gsv"), key=lambda p: int(p.stem.split("_")[-1]))
    days, above = [], []
    for path in files:
        text = path.read_text(errors="replace")
        hit = re.search(r"T\s*=\s*([0-9.eE+-]+)\s*days", text.splitlines()[2])
        days.append(0.0 if hit is None else float(hit.group(1)))   # snapshot 0 is "initial"
        df = parse_transient(text, str(path))
        z = df["z"].to_numpy()
        depth, dz = (z.max() - z) * 100.0, np.abs(np.gradient(z))
        vf = df[CALCITE].to_numpy() * dz                           # m3 of calcite per m2 of column
        above.append({cp: float(vf[depth <= cp].sum()) for cp in PLANES})

    days = np.asarray(days)
    assert days[0] == 0.0 and (np.diff(days) > 0).all(), f"{folder}: snapshots out of order"
    assert days[-1] >= RUN_DAYS - 1e-6, f"{folder}: snapshots stop at day {days[-1]}"
    return {cp: np.interp(DAYS, days, [(above[0][cp] - a[cp]) / V_CALCITE for a in above])
            for cp in PLANES}


cmin = {}
for run in RUNS:
    cmin[run] = calcite_dissolved(*run)
    print(f"  {run[0]:14s} {run[1]:9s} {run[2]:5s} {cmin[run][PLANE][-1]:8.3f} mol/m2 "
          f"above {PLANE} cm", flush=True)

  Cecil          hourly    ctrl     0.000 mol/m2 above 100 cm
  Cecil          hourly    erw      0.000 mol/m2 above 100 cm
  Cecil          daily     ctrl     0.000 mol/m2 above 100 cm
  Cecil          daily     erw      0.000 mol/m2 above 100 cm
  Cecil          monthly   ctrl     0.000 mol/m2 above 100 cm
  Cecil          monthly   erw      0.000 mol/m2 above 100 cm
  Cecil          longterm  ctrl     0.000 mol/m2 above 100 cm
  Cecil          longterm  erw      0.000 mol/m2 above 100 cm
  Flanagan       hourly    ctrl     8.506 mol/m2 above 100 cm
  Flanagan       hourly    erw      6.855 mol/m2 above 100 cm
  Flanagan       daily     ctrl     8.443 mol/m2 above 100 cm
  Flanagan       daily     erw      6.775 mol/m2 above 100 cm
  Flanagan       monthly   ctrl     8.918 mol/m2 above 100 cm
  Flanagan       monthly   erw      6.733 mol/m2 above 100 cm
  Flanagan       longterm  ctrl     9.578 mol/m2 above 100 cm
  Flanagan       longterm  erw      7.127 mol/m2 above 100 cm
  Housto

## 5. Forsterite in the column

`forst-ph` in `<case>_o.mms`, the domain-integrated mineral inventory. The domain is 1 m2 in plan,
so mol is mol/m2. Only the `erw` runs carry feedstock.

In [7]:
def forsterite_moles(site, case):
    """Forsterite in the domain (mol) on each day in `DAYS`."""
    path = f"{run_key(site, case, 'erw')}/{case}_o.mms"
    df = parse_transient(fs.cat_file(path).decode("latin1"), path)
    t, m = dedupe(df["time"].to_numpy(float), df[MINERAL].to_numpy(float))
    assert t[0] == 0.0 and t[-1] >= RUN_DAYS - 1e-6, f"{path} does not span 0-{RUN_DAYS} d"
    return np.interp(DAYS, t, m)


pairs = [(s, c) for s in SITES for c in CASES]
with ThreadPoolExecutor(WORKERS) as pool:
    moles = dict(zip(pairs, pool.map(lambda r: forsterite_moles(*r), pairs)))

forsterite_mass = np.array([[moles[(s, c)] for c in CASES] for s in SITES]) * MOL_TO_T_HA

inventory0 = {}
for site in SITES:
    m0 = {c: moles[(site, c)][0] for c in CASES}
    assert len(set(m0.values())) == 1, f"{site}: cases disagree on the feedstock applied, {m0}"
    assert m0[CASES[0]] > 0, f"{site} has no forsterite at day 0"
    inventory0[site] = m0[CASES[0]]

cdr_potential = np.array([inventory0[s] * EQ_PER_MOL_FORST * EQ_TO_T_CO2_HA for s in SITES])

print(f"applied: {inventory0[SITES[0]] * MOL_TO_T_HA:.3f} t/ha forsterite, "
      f"ceiling {cdr_potential[0]:.2f} t CO2/ha")
print("\nforsterite left at day 3650 (t/ha):")
print(pd.DataFrame(forsterite_mass[..., -1], index=SITES, columns=CASES).round(3))

applied: 9.833 t/ha forsterite, ceiling 12.30 t CO2/ha

forsterite left at day 3650 (t/ha):
              hourly  daily  monthly  longterm
Cecil          0.545  0.541    0.444     0.385
Flanagan       0.512  0.509    0.397     0.353
HoustonBlack   2.306  2.243    1.738     1.450
Kalamazoo      0.101  0.098    0.025     0.009
Kuma           4.140  4.134    3.959     3.421
Palouse        1.396  1.422    1.271     1.163
Pullman        5.299  5.313    5.400     5.560
Yolo           2.409  2.414    2.055     1.857


## 6. CDR

The difference of the two treatments, in t CO2/ha, less the carbonate term.

In [8]:
shape = (len(SITES), len(CASES), len(PLANES), DAYS.size)
cdr_alk = np.full(shape, np.nan)
carbonate_c = np.full(shape, np.nan)
alk_export = {t: np.full(shape, np.nan) for t in TREATMENTS}
drainage = {t: np.full(shape, np.nan) for t in TREATMENTS}
for i, site in enumerate(SITES):
    for j, case in enumerate(CASES):
        for k, cp in enumerate(PLANES):
            cdr_alk[i, j, k] = (export[(site, case, "erw")][cp]
                                - export[(site, case, "ctrl")][cp]) * EQ_TO_T_CO2_HA
            carbonate_c[i, j, k] = (cmin[(site, case, "erw")][cp]
                                    - cmin[(site, case, "ctrl")][cp]) * EQ_TO_T_CO2_HA
            for treat in TREATMENTS:
                alk_export[treat][i, j, k] = export[(site, case, treat)][cp] * EQ_TO_T_CO2_HA
                drainage[treat][i, j, k] = drained[(site, case, treat)][cp]

cdr = cdr_alk - CARBON_PER_MOL_CALCITE * carbonate_c

P = PLANES.index(PLANE)
print(f"10-year CDR at {PLANE} cm (t CO2/ha):")
print(pd.concat({"uncorrected": pd.DataFrame(cdr_alk[:, :, P, -1], index=SITES, columns=CASES),
                 "correction": pd.DataFrame(-carbonate_c[:, :, P, -1], index=SITES, columns=CASES),
                 "corrected": pd.DataFrame(cdr[:, :, P, -1], index=SITES, columns=CASES)},
                axis=1).round(3))

10-year CDR at 100 cm (t CO2/ha):
             uncorrected                         correction                 \
                  hourly  daily monthly longterm     hourly  daily monthly   
Cecil              0.560  0.571   0.794    0.970     -0.000 -0.000  -0.000   
Flanagan           0.275  0.200   0.309    0.365      0.727  0.734   0.962   
HoustonBlack      -0.521 -0.607  -0.690   -0.734      4.957  5.020   5.390   
Kalamazoo          0.279  0.288   0.376    0.419     -0.000 -0.000  -0.000   
Kuma               0.327  0.306   0.216   -0.052      2.279  2.288   2.397   
Palouse            2.205  2.185   2.490    2.718     -0.000 -0.000  -0.000   
Pullman           -0.712 -0.728  -0.867   -1.512      2.805  2.794   2.758   
Yolo               0.795  0.750   0.487    0.326      1.272  1.315   1.742   

                      corrected                          
             longterm    hourly  daily monthly longterm  
Cecil          -0.000     0.560  0.571   0.794    0.970  
Flanagan   

## 7. Assemble

In [9]:
dims = ("site", "case", "plane", "time")

ds = xr.Dataset(
    {
        "cdr": (dims, cdr.astype(np.float32)),
        "cdr_alk": (dims, cdr_alk.astype(np.float32)),
        "carbonate_c": (dims, carbonate_c.astype(np.float32)),
        "alk_export_ctrl": (dims, alk_export["ctrl"].astype(np.float32)),
        "alk_export_erw": (dims, alk_export["erw"].astype(np.float32)),
        "drainage_ctrl": (dims, drainage["ctrl"].astype(np.float32)),
        "drainage_erw": (dims, drainage["erw"].astype(np.float32)),
        "forsterite_mass": (("site", "case", "time"), forsterite_mass.astype(np.float32)),
        "cdr_potential": (("site",), cdr_potential.astype(np.float32)),
    },
    coords={
        "site": SITES,
        "case": CASES,
        "plane": np.array(PLANES, dtype=np.int32),
        "time": DAYS,
        "state": ("site", [states_per_site[s] for s in SITES]),
    },
)

UNITS = "t CO2/ha"
BASIS = ("cumulative alkalinity export across a control plane, at 1 mol CO2 per equivalent "
         "and 44.01 g/mol; the reconstruction is described in the notebook header")

ds["cdr"].attrs = {
    "units": UNITS, "long_name": "cumulative CDR, carbonate-corrected",
    "description": f"`cdr_alk` minus `carbonate_c`, where `cdr_alk` is ERW minus CTRL {BASIS}",
    "sign_note": "can be negative -- the amendment can suppress export relative to its own control",
    "depth_note": ("NOT monotone in `plane` -- a deeper plane can carry more than a shallower one, "
                   "because the displaced base-cation wave runs ahead of the Mg front."),
    "cell_methods": "time: point"}
ds["cdr_alk"].attrs = {
    "units": UNITS, "long_name": "cumulative CDR, uncorrected",
    "description": (f"ERW minus CTRL {BASIS}. Credits carbonate-derived alkalinity at 1 mol "
                    "CO2/eq, thus not used for CDR but kept for comparison only"),
    "cell_methods": "time: point"}
ds["carbonate_c"].attrs = {
    "units": UNITS, "long_name": "carbonate carbon released above the plane, ERW minus CTRL",
    "description": ("one mole of CO2 per mole of calcite dissolved above the plane, from the "
                    f"`{CALCITE}` volume fraction in the `.gsv` snapshots over its molar volume "
                    f"({V_CALCITE * 1e6:.3f} cm3/mol), differenced between the treatments. "
                    "`cdr = cdr_alk - carbonate_c`"),
    "sign_note": ("negative where the amended run net precipitates calcite, "
                  "which returns CO2 and raises `cdr`"),
    "zero_note": "exactly zero at sites with no soil carbonate above the plane",
    "cell_methods": "time: point"}
ds["alk_export_ctrl"].attrs = {
    "units": UNITS, "long_name": "cumulative alkalinity export, control run",
    "description": f"the unamended baseline: {BASIS}"}
ds["alk_export_erw"].attrs = {
    "units": UNITS, "long_name": "cumulative alkalinity export, amended run",
    "description": f"the amended run: {BASIS}. `cdr_alk` is this minus `alk_export_ctrl`"}
for treat in TREATMENTS:
    ds[f"drainage_{treat}"].attrs = {
        "units": "m", "long_name": f"cumulative water past the plane, {treat} run",
        "description": ("net downward water flux integrated over time")}
ds["forsterite_mass"].attrs = {
    "units": "t/ha", "long_name": "feedstock forsterite remaining in the soil column",
    "description": (f"`{MINERAL}` in `<case>_o.mms`, the domain-integrated inventory, x "
                    f"{MOLAR_MASS} g/mol; the domain is 1 m2 in plan. The `erw` runs only -- the "
                    "controls carry no feedstock. Its day-0 value is the amendment as applied, "
                    "and applied minus this is the mass dissolved."),
    "cell_methods": "time: point"}
ds["cdr_potential"].attrs = {
    "units": UNITS, "long_name": "CDR at complete conversion of the applied feedstock",
    "description": ("forsterite at day 0 x 4 eq/mol, at 1 mol CO2 per eq -- silicate alkalinity")}

ds["site"].attrs = {"long_name": "soil series"}
ds["case"].attrs = {
    "long_name": "meteorological forcing resolution",
    "description": ("resolution the 10-year met record was resampled to before forcing the run; "
                    "the runs are otherwise identical, sharing one spin-up per site")}
ds["plane"].attrs = {
    "units": "cm", "long_name": "depth of the control plane below ground",
    "positive": "down",
    "description": ("the four depths where the deck monitors both straddling cells")}
ds["time"].attrs = {"units": "days", "long_name": "days since the amendment",
                    "description": "day ends, 0 to 3650; no leap days in this calendar"}
ds["state"].attrs = {"long_name": "site location"}

ds.attrs = {
    "title": "Cumulative CDR by site, met forcing resolution and control plane",
    "summary": ("ERW minus CTRL cumulative alkalinity export across soil control planes in the "
                "MIN3P met_forcing_rxn simulations, corrected for soil carbonate and expressed "
                "as tonnes of CO2 per hectare, with the feedstock inventory left in the column"),
    "carbonate_corrected": True,
    "carbonate_correction": ("`cdr` = `cdr_alk` - `carbonate_c`: one mole of CO2 per mole of "
                             "calcite dissolved above the plane, ERW minus CTRL."),
    "created_by": "figures/postprocess-data/create-site-co2.ipynb",
    "feedstock": "forsterite, 9.833 t/ha over 1-29 cm, identical at all eight sites",
    "eq_to_t_co2_ha": EQ_TO_T_CO2_HA,
    "molar_mass_forsterite_g_per_mol": MOLAR_MASS,
    "molar_volume_calcite_m3_per_mol": V_CALCITE,
}

ds

<xarray.Dataset> Size: 14MB
Dimensions:          (site: 8, case: 4, plane: 4, time: 3651)
Coordinates:
  * site             (site) <U12 384B 'Cecil' 'Flanagan' ... 'Pullman' 'Yolo'
    state            (site) <U11 352B 'Northern SC' ... 'Central CA'
  * case             (case) <U8 128B 'hourly' 'daily' 'monthly' 'longterm'
  * plane            (plane) int32 16B 50 100 200 300
  * time             (time) float64 29kB 0.0 1.0 2.0 ... 3.649e+03 3.65e+03
Data variables:
    cdr              (site, case, plane, time) float32 2MB 0.0 0.0 ... 2.254
    cdr_alk          (site, case, plane, time) float32 2MB 0.0 0.0 ... 0.3358
    carbonate_c      (site, case, plane, time) float32 2MB 0.0 0.0 ... -1.918
    alk_export_ctrl  (site, case, plane, time) float32 2MB 0.0 ... 6.93
    alk_export_erw   (site, case, plane, time) float32 2MB 0.0 ... 7.266
    drainage_ctrl    (site, case, plane, time) float32 2MB 0.0 ... 6.987
    drainage_erw     (site, case, plane, time) float32 2MB 0.0 ... 6.987
    forsterite_mass  (site, case, time) float32 467kB 9.833 9.825 ... 1.857
    cdr_potential    (site) float32 32B 12.3 12.3 12.3 12.3 12.3 12.3 12.3 12.3
Attributes:
    title:                            Cumulative CDR by site, met forcing res...
    summary:                          ERW minus CTRL cumulative alkalinity ex...
    carbonate_corrected:              True
    carbonate_correction:             `cdr` = `cdr_alk` - `carbonate_c`: one ...
    created_by:                       figures/postprocess-data/create-site-co...
    feedstock:                        forsterite, 9.833 t/ha over 1-29 cm, id...
    eq_to_t_co2_ha:                   0.44009999999999994
    molar_mass_forsterite_g_per_mol:  140.6931
    molar_volume_calcite_m3_per_mol:  3.6933357933579335e-05

## 8. Checks

In [10]:
# The cumulative terms start at zero, and the feedstock starts at what was applied.
for name in ["cdr", "cdr_alk", "carbonate_c", "alk_export_ctrl", "alk_export_erw",
             "drainage_ctrl", "drainage_erw"]:
    assert np.all(ds[name].isel(time=0).to_numpy() == 0), f"{name} is not zero at t = 0"
assert np.allclose(forsterite_mass[..., 0],
                   np.array([inventory0[s] for s in SITES])[:, None] * MOL_TO_T_HA)

# Feedstock only dissolves. `_o.mms` writes 5 significant figures on a ~7 mol inventory, so a
# ~1e-4 mol wobble on a flat record is the reporting quantum, not re-precipitation.
QUANTUM = 2.5e-4 * MOL_TO_T_HA
rebound = float(np.diff(forsterite_mass, axis=-1).max())
assert rebound <= QUANTUM, f"forsterite re-precipitates by {rebound:.1e} t/ha"
print(f"largest increase in forsterite_mass: {rebound:.1e} t/ha (quantum {QUANTUM:.1e})")

# Gross export steps backwards on days when water is drawn up across the plane; check that is
# what the backward steps are, and that they are small.
back_n = coincident = 0
back_worst = 0.0
for treat in TREATMENTS:
    da = np.diff(alk_export[treat], axis=-1)
    neg = da < -1e-12
    back_n += int(neg.sum())
    coincident += int((neg & (np.diff(drainage[treat], axis=-1) < 0)).sum())
    rel = np.abs(np.where(neg, da, 0.0)).max(axis=-1) / np.abs(alk_export[treat][..., -1])
    back_worst = max(back_worst, float(rel.max()))
share = 100 * coincident / back_n if back_n else 100.0
print(f"backward steps in gross export: {back_n} of {2 * cdr[..., 1:].size}; "
      f"{share:.2f}% coincide with water moving up, largest {100 * back_worst:.3f}% of the total")
assert share > 99.0, "export runs backwards on days when water does not"
assert back_worst < 1e-3, "a backward step is more than 0.1% of the ten-year total"

# The three CDR variables close on each other and on the gross terms.
assert np.allclose(cdr_alk, alk_export["erw"] - alk_export["ctrl"], atol=1e-12)
assert np.allclose(cdr, cdr_alk - CARBON_PER_MOL_CALCITE * carbonate_c, atol=1e-12)

# No site exports more alkalinity than the feedstock could ever supply. The bound is on the
# alkalinity term: `cdr` can exceed it where the amended run precipitates calcite, which stores
# carbon the forsterite ceiling does not account for.
assert (cdr_alk.max(axis=(1, 2, 3)) <= cdr_potential + 1e-6).all(), "export exceeds the ceiling"

untouched = [s for i, s in enumerate(SITES) if np.abs(carbonate_c[i, :, P]).max() < 1e-9]
print(f"\nno carbonate above {PLANE} cm: {', '.join(untouched) or 'none'}")
print(f"\n10-year CDR at {PLANE} cm as a % of the {cdr_potential[0]:.2f} {UNITS} ceiling:")
print((100 * pd.DataFrame(cdr[:, :, P, -1], index=SITES, columns=CASES)
       .div(cdr_potential, axis=0)).round(1))
print("\n% of the feedstock dissolved after 10 years:")
print((100 * (1 - pd.DataFrame(forsterite_mass[..., -1], index=SITES, columns=CASES)
              .div(forsterite_mass[:, 0, 0], axis=0))).round(1))

largest increase in forsterite_mass: -7.8e-05 t/ha (quantum 3.5e-04)
backward steps in gross export: 38262 of 934400; 99.81% coincide with water moving up, largest 0.038% of the total

no carbonate above 100 cm: Cecil, Palouse

10-year CDR at 100 cm as a % of the 12.30 t CO2/ha ceiling:
              hourly  daily  monthly  longterm
Cecil            4.6    4.6      6.5       7.9
Flanagan         8.1    7.6     10.3      11.7
HoustonBlack    36.1   35.9     38.2      39.4
Kalamazoo        2.3    2.3      3.1       3.4
Kuma            21.2   21.1     21.2      22.7
Palouse         17.9   17.8     20.2      22.1
Pullman         17.0   16.8     15.4      11.9
Yolo            16.8   16.8     18.1      18.5

% of the feedstock dissolved after 10 years:
              hourly  daily  monthly  longterm
Cecil           94.5   94.5     95.5      96.1
Flanagan        94.8   94.8     96.0      96.4
HoustonBlack    76.6   77.2     82.3      85.3
Kalamazoo       99.0   99.0     99.7      99.9
Kuma    

## 9. Write

In [11]:
NAME = "site_cdr"
url = f"{OUT_ROOT}/{NAME}.zarr"

if WRITE:
    if not OVERWRITE and fs.exists(url.replace("s3://", "")):
        raise FileExistsError(f"{url} exists and OVERWRITE is False")
    # A couple of megabytes an array, so one chunk each. Zarr v2: under v3 the string coords land
    # on fixed-length UTF-32, which zarr-python warns is unportable.
    ds.chunk({d: n for d, n in ds.sizes.items()}).to_zarr(
        url, mode="w", zarr_format=2, consolidated=True)
    print("wrote", url)
else:
    print(f"WRITE is False -- would write {url}  {dict(ds.sizes)}")

wrote s3://carbonplan-carbon-removal/ew-workflows-data/min3p/simulations/postprocessed/met_forcing_rxn/site_cdr.zarr


## 10. Read back

In [12]:
if WRITE:
    back = xr.open_zarr(url).load()
    xr.testing.assert_allclose(back, ds)
    mb = sum(f["size"] for f in fs.find(url.replace("s3://", ""), detail=True).values()) / 1e6
    print(f"{NAME:12s} {str(dict(back.sizes)):<56s} {mb:6.3f} MB")
    print("round-trips\n")
    print(f"carbonate_corrected: {back.attrs['carbonate_corrected']}")
    print(f"\n10-year corrected CDR at {PLANE} cm ({UNITS}):")
    print(back["cdr"].sel(plane=PLANE).isel(time=-1).to_pandas().round(3))
    print("\nforsterite left at day 3650 (t/ha):")
    print(back["forsterite_mass"].isel(time=-1).to_pandas().round(3))

site_cdr     {'site': 8, 'case': 4, 'plane': 4, 'time': 3651}          7.733 MB
round-trips

carbonate_corrected: True

10-year corrected CDR at 100 cm (t CO2/ha):
case          hourly  daily  monthly  longterm
site                                          
Cecil          0.560  0.571    0.794     0.970
Flanagan       1.002  0.934    1.270     1.444
HoustonBlack   4.437  4.413    4.700     4.848
Kalamazoo      0.279  0.288    0.376     0.419
Kuma           2.606  2.593    2.613     2.790
Palouse        2.205  2.185    2.490     2.718
Pullman        2.093  2.066    1.892     1.459
Yolo           2.067  2.065    2.230     2.278

forsterite left at day 3650 (t/ha):
case          hourly  daily  monthly  longterm
site                                          
Cecil          0.545  0.541    0.444     0.385
Flanagan       0.512  0.509    0.397     0.353
HoustonBlack   2.306  2.243    1.738     1.450
Kalamazoo      0.101  0.098    0.025     0.009
Kuma           4.140  4.134    3.959     3.421


In [ ]:
# ---